# Section 1: Setup & Imports
- torch, transformers, peft, plotly, gradio, captum

In [1]:
import torch
import transformers
import peft
import plotly.io as pio
from data_loader import get_dataloaders
from models import ZeroShotCLIPAnalyzer, CLIPMultimodalClassifier, setup_lora_model
from train import train_eval_loop
from visualize import *
from interpretability import *
from benchmark import *
from sklearn.metrics import f1_score
from transformers import CLIPProcessor

pio.templates.default = "plotly_white"

/home/ml4u/miniconda3/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


# Section 2: Dataset Loading & EDA
- Load mmimdb from HuggingFace
- **Plotly**: Genre distribution (interactive bar chart)

In [2]:
batch_size = 64
train_loader, val_loader, test_loader, num_genres, genre_names = get_dataloaders("openai/clip-vit-base-patch32", batch_size=batch_size, k_shot=32)
print("Detected Genres:", genre_names)

# Compute Genre Distribution for Training Set
# For simplicity in EDA, we sum up the labels from our train_loader
genre_counts = torch.zeros(num_genres)
for batch in train_loader:
    genre_counts += batch['labels'].sum(dim=0)

fig_dist = plot_genre_distribution(genre_counts.numpy(), genre_names)
fig_dist.write_html("genre_distribution.html")
# fig_dist.show()

Loading data processor...
Loading mmimdb dataset (may take some time)...
Detected 23 genres.


/home/ml4u/miniconda3/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (97200000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ml4u/miniconda3/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (97888494 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ml4u/miniconda3/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (174960000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ml4u/miniconda3/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (99331758 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/ml4u/miniconda3/lib/python3.12/site-packages/PIL/Image.py:3402: DecompressionBombWarning: Image size (94770000 pixels) exceeds li

Detected Genres: ['Drama', 'Comedy', 'Romance', 'Thriller', 'Crime', 'Action', 'Adventure', 'Horror', 'Documentary', 'Mystery', 'Sci-Fi', 'Fantasy', 'Family', 'Biography', 'War', 'History', 'Music', 'Musical', 'Sport', 'Western', 'Short', 'Film-Noir', 'Adult']


# Section 3: Data Preprocessing & Dataloaders
Images are preprocessed and text is tokenized dynamically via HuggingFace's CLIPProcessor within data_loader.py. K-shot sampling has been performed producing loaders.

# Section 4: Zero-Shot Classification (CLIP)
- Load pretrained CLIP
- Evaluate on test set
- **Plotly**: Per-genre performance

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
zero_shot_model = ZeroShotCLIPAnalyzer(device=device)

# Using val_loader or test_loader for zero shot
# For speed we can run on a subset or full test_loader
print("Running Zero-Shot Prediction...")
zs_probs, zs_labels = zero_shot_model.predict(test_loader, genre_names)

# Metric Computation
zs_preds = (zs_probs > 0.5).numpy()
zs_labels_np = zs_labels.numpy()
zs_f1 = f1_score(zs_labels_np, zs_preds, average='micro')
print("Zero Shot F1 Score:", zs_f1)

Running Zero-Shot Prediction...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Zero Shot F1 Score: 0.05626223091976516


# Section 5: LoRA Few-Shot Classification
- Define CLIP + classification head architecture
- Apply LoRA config (r=8, alpha=16)

In [4]:
print("Setting up LoRA model...")
base_classifier = CLIPMultimodalClassifier(num_genres=num_genres).to(device)
lora_model = setup_lora_model(base_classifier)
lora_model.print_trainable_parameters()

# Train K=32 
history = train_eval_loop(lora_model, train_loader, val_loader, device=device, num_epochs=10) # 10 for demonstration, up to 20/30 in practice

Setting up LoRA model...
trainable params: 2,748,439 || all params: 154,562,351 || trainable%: 1.7782


Epoch 1/10 [Train]:   0%|          | 0/7 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZER

OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 50.88 MiB is free. Process 2608256 has 15.31 GiB memory in use. Including non-PyTorch memory, this process has 7.51 GiB memory in use. Of the allocated memory 6.87 GiB is allocated by PyTorch, and 203.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# Section 6: Results Comparison
Plot Training curves and zero-shot vs Few-Shot.

In [ ]:
fig_train = plot_training_progress(history)
fig_train.write_html("training_progress.html")
# fig_train.show()

# Metric comparison
metrics_comparison = {
    'Zero-Shot CLIP': zs_f1,
    'LoRA K=32': max(history['val_f1'])
}
fig_comp = plot_metrics_comparison(metrics_comparison)
fig_comp.write_html("metrics_comparison.html")
# fig_comp.show()

# Section 7: Interpretability
- Attention Rollout for image
- Integrated Gradients for text

In [ ]:
# Fetch a single batch for visualization
batch = next(iter(val_loader))
pixel_values = batch["pixel_values"][0:1].to(device)
raw_image = Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8))
raw_text = "A visually stunning movie"

print("Computing Attention map...")
# For interpretability:
# lora_model base is CLIPModel at lora_model.base_model.model.clip
# We can use that for interpretability.
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Un-comment the execution below after ensuring raw_image and raw_text are correctly returned in datasets or just manually loading an image from the dataset.
# attention_map = get_image_attention_map(lora_model.base_model.model.clip, processor, Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8)))
# tokens, text_attributions = compute_text_attribution(lora_model.base_model.model.clip, processor, "A great movie with plot twist")

# Section 8: Gradio Demo
- App code located in `app.py`

# Section 9: Model Export & Benchmarking
ONNX Export and benchmarking.

In [ ]:
print("Exporting ZeroShot model to ONNX...")
onnx_path = export_to_onnx(zero_shot_model.model, "zero_shot_clip", device)
try:
    metrics_pt, lats_pt = benchmark_pytorch(zero_shot_model.model, test_loader, device=device, n_samples=300)
    metrics_ort, lats_ort = benchmark_onnx(onnx_path, test_loader, n_samples=300)

    fig_lat = plot_latency_distribution({'PyTorch': lats_pt, 'ONNX': lats_ort})
    fig_lat.write_html("latency_distribution.html")
    # fig_lat.show()
except Exception as e:
    print("Could not run ONNX benchmarking. Error:", e)

# Section 10: Conclusion
We successfully integrated a Zero-Shot classification approach, fine-tuned a LoRA multi-modal model, demonstrated performance comparisons, generated Plotly charts, explored attention interpretability, and packaged it inside a Gradio UI.